# IPOVision — 01: Data Source Testing & Verification

**Phase:** Phase 1 — Data Source Testing  
**Objective:** Test whether public data sources for Indian Mainboard IPOs can be accessed reliably, identify technical limitations, and establish a multi-source collection strategy.

---

## 1. Setup & Imports

We use the project's defensive `IPOHttpClient` which enforces:
- Browser User-Agent headers
- Rate limiting between requests (2.0s minimum delay)
- Retry adapters for transient 5xx errors
- Structured response dictionaries

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import json

# Navigate to project root
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.collection.http_client import IPOHttpClient
from src.collection.investorgain_scraper import InvestorGainScraper
from src.collection.chittorgarh_detail_scraper import ChittorgarhDetailScraper

client = IPOHttpClient(rate_limit_delay=2.0)
print(f"Project root: {PROJECT_ROOT}")
print("IPOHttpClient initialized successfully.")

## 2. Load and Review Automated Source Test Results

In Phase 1, we tested 10 different public endpoints across 5 core categories:
1. **IPO Listing Performance** (Exchange and aggregator summary tables)
2. **IPO Detail Pages** (Issue structure, fundamentals, KPIs, valuation)
3. **Grey Market Premium (GMP)** (Market sentiment quotes and historical trends)
4. **Subscription Bidding Data** (QIB, NII, Retail multiples)
5. **RHP/DRHP Offer Documents** (SEBI statutory filings in PDF)

In [ ]:
report_path = PROJECT_ROOT / "data" / "raw" / "source_test_report.json"
with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print(f"Total sources tested: {report['total_sources_tested']}")
print(f"Accessible: {report['successful']} | Failed: {report['failed']}")

# Display as a clean DataFrame summary
summary_data = []
for r in report["results"]:
    summary_data.append({
        "Source": r["source_name"],
        "Category": r["category"],
        "HTTP Status": r["http_status"],
        "Response (ms)": round(r["response_time_ms"], 0) if r["response_time_ms"] else None,
        "Page Size (KB)": round(r["page_size_kb"], 0) if r["page_size_kb"] else None,
        "Tables": r["tables_found"],
        "Key Fields Detected": len(r["data_fields_found"]),
    })

df_summary = pd.DataFrame(summary_data)
df_summary

## 3. Key Discovery: Static HTML vs Client-Side (React) Rendering

Our deep inspection revealed a fundamental technical distinction among sources:

| Source Page Type | Rendering Method | Scraping Strategy | Reliability |
| :--- | :--- | :--- | :--- |
| **Chittorgarh Summary Reports** (`/report/...`) | React / Client-Side JS | Tables rendered dynamically via JS; 0 `<table>` tags in static HTML. | Requires browser automation or alternative source |
| **Chittorgarh Detail Pages** (`/ipo/...`) | Server-Side HTML | **14–16 full HTML tables** with multi-year financials, KPIs, valuation, and listing prices. | **HIGH — Primary source for fundamentals** |
| **InvestorGain Listing Tables** (`/report/live-ipo-gmp/...`) | Server-Side HTML | Clean `table#reportTable` with live issue price, P/E, size, and **full subscription breakdown**. | **HIGH — Primary source for subscription & listing** |
| **SEBI Offer Documents** (`sebi.gov.in`) | Server-Side HTML | Direct links to statutory RHP/DRHP PDFs. | **HIGH — Primary source for prospectus RAG** |

## 4. Test Proof-of-Concept Scraper: InvestorGain (Listing & Subscriptions)

Let's test our working `InvestorGainScraper` to extract live Mainboard IPO listing and subscription records.

In [ ]:
ig_scraper = InvestorGainScraper(client=client)
listing_records = ig_scraper.fetch_listing_performance()

df_listings = pd.DataFrame(listing_records)
print(f"Extracted {len(df_listings)} Mainboard IPO records:")
df_listings[["company_name", "issue_price", "pe_ratio", "overall_subscription", "qib_subscription", "nii_subscription", "retail_subscription", "closing_date"]].head(10)

## 5. Test Proof-of-Concept Scraper: Chittorgarh (Company Fundamentals)

Now let's test our `ChittorgarhDetailScraper` which parses complex multi-table IPO detail pages into structured fundamental metrics.

In [ ]:
chit_scraper = ChittorgarhDetailScraper(client=client)
detail_data = chit_scraper.parse_detail_page("https://www.chittorgarh.com/ipo/tata-technologies-ipo/1577/")

print(f"Company: {detail_data.get('company_name')}")
print("\n--- Issue Details ---")
for k, v in detail_data.get("issue_details", {}).items():
    print(f"  {k}: {v}")

print("\n--- Multi-Year Financial Trends ---")
df_fin = pd.DataFrame(detail_data.get("financials_multi_year", []))
print(df_fin.to_string(index=False))

print("\n--- Key Performance Indicators (KPIs) ---")
for k, v in detail_data.get("kpis", {}).items():
    print(f"  {k}: {v}")

print("\n--- Valuation Multiples ---")
for k, v in detail_data.get("valuation", {}).items():
    print(f"  {k}: {v}")

print("\n--- Listing Day Prices ---")
for k, v in detail_data.get("listing_details", {}).items():
    print(f"  {k}: {v}")

## 6. Category E: RHP/DRHP Prospectus Document Inspection

We verified that statutory offer documents from SEBI:
1. Can be discovered and downloaded directly via HTTP.
2. Contain **100% selectable digital text** (0 scanned/image pages in our sample).
3. Average **528 words per page**, making them ideal for chunking and vector storage in Phase 5.
4. Contain standard section headings: `Risk Factors`, `Financial Information`, `Promoter Information`.

In [ ]:
import pymupdf

sample_pdf_path = PROJECT_ROOT / "data" / "raw" / "sample_prospectus.pdf"
if sample_pdf_path.exists():
    doc = pymupdf.open(sample_pdf_path)
    print(f"Loaded sample PDF: {sample_pdf_path.name}")
    print(f"Pages: {len(doc)}")
    print(f"File size: {sample_pdf_path.stat().st_size / 1024:.0f} KB")
    
    # Show snippet of Page 2 (typically Risk Factors)
    print("\nPage 2 Excerpt (first 400 characters):")
    print("-" * 50)
    print(doc[1].get_text()[:400])
    print("-" * 50)
    doc.close()
else:
    print("Sample PDF not found. Run test_real_pdf.py first.")

## 7. Phase 1 Conclusions & Multi-Source Strategy

### Final Source Allocation for Phase 2:
1. **IPO Information & Outcomes:** InvestorGain + Chittorgarh Detail Pages provide verified issue prices, dates, lot sizes, and listing open/close prices.
2. **Subscription Multiples:** InvestorGain provides complete historical QIB, NII, and Retail subscription data without paywalls.
3. **Company Fundamentals & KPIs:** Chittorgarh detail pages provide 4-year restated financial statements, valuation metrics, and balance sheet numbers.
4. **GMP Sentiment:** InvestorGain and Chittorgarh detail pages provide point-in-time GMP observations with timestamp attribution.
5. **RHP/DRHP Intelligence:** SEBI statutory filings provide downloadable, text-extractable PDFs for the RAG engine.

**Phase 1 Deliverables Complete:** All sources tested, technical limitations documented, and 2 working proof-of-concept scrapers operational.